<a href="https://colab.research.google.com/github/lhg96/-AI-RSSCrawler/blob/main/6_%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D_%EC%A7%80%EB%8F%84%ED%95%99%EC%8A%B5_%EB%B6%84%EB%A5%98_AutoML%EC%9D%84_%EC%9D%B4%EC%9A%A9%ED%95%9C_%EC%B5%9C%EC%A0%81%EB%AA%A8%EB%8D%B8_%EC%84%A0%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoML

- 데이터 수집/분석-EDA등을 제외한 전 공정을 AutoML로 진행 가능함
- 파이캐럿 사용

# 설치

In [ ]:
!pip install pycaret[full]
# 세션 다시 시작 메세지 발생 -> 다시 시작 클릭
!pip install -U jinja2      # 템플릿엔진(웹에서 사용하는)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 3.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of category-encoders to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 11.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 56.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of flask to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 3.

In [ ]:
import pycaret

In [ ]:
# 써드 파트 알고리즘 설치
!pip install catboost -q

# 베이스라인 구축(간단하게)

- 데이터
  - load_breast_cancer()
- 알고리즘
  - 랜덤포레스트
- 데이터
  - 9:1 분할 (소량의 데이터에서 훈련량을 높이는 방향성)
  - 난수 시드 사용
  - 층화 x
- 평가
  - 정확도

In [ ]:
# 1. 알고리즘 선택
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

# 2. 데이터 준비및 분할
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()

# 파이캐럿 사용시 편의성 고려하여 df로 구성함
import pandas as pd
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target # 정답 데이터 추가
df.head(2)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0


In [ ]:
df.shape

(569, 31)

In [ ]:

# 피처 데이터만 추출 => df.iloc[ 1차원 (행) , 2차원 (열) ] => 데이터프레임의 데이터 추출방법
# : => 처음부터 끝까지
# :-1 => 처음부터 마지막 1개를 제외한 나머지 모두=
df.iloc[ : , :-1 ]

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


- 데이터 추출법
  - 적용 자료구조
    - 문자열, 리스트, 튜플, 딕셔너리, 배열(넘파이), 시리즈, 데이터프레임(분석, 머신러닝, 딥러닝), 텐서(딥러닝)
  - 인덱싱
    - 차원 축소, 데이터 특정한다(인덱스)
  - 슬라이싱
    - 차원 유지, 데이터를 자른다(연속적, 이웃한 데이터)

  - 인덱스
    - 정방향 => 0, 1, 2, .. 앞에서부터 체크
    - 역방향 => -1, -2, -3, ... 뒤에서부터 체크


In [ ]:
# 정답 데이터만 추출 => Series로 출력됨(1차원)
# : => 처음부터 끝까지
# -1 => 역방향 인덱스 마지막 컬럼 1개만 특정 => 차원축소
df.iloc[ : , -1 ] # 모든 데이터에 정답(컬럼명 target)을 추출하시오

,target
0,0
1,0
2,0
3,0
4,0
...,...
564,0
565,0
566,0
567,0


In [ ]:
# 데이터 분할 처리 (9:1)
from sklearn.model_selection import train_test_split

# 피처 데이터 (정답 제외한 나머지 모든 데이터만 추출하시오)
X = df.iloc[ : , :-1 ] # 1차원 데이터는 모두 가져온다, 2차원데이터는 마지막 1개만 제외
# 정답 데이터
y = df.iloc[ : , -1 ] # 1차원 데이터는 모두 가져온다, 2차원 데이터는 정답만(마지막 1개만)추출

# 데이터 분할 처리, 9:1
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                   test_size=0.1, random_state=42)

In [ ]:
# 3. 훈련
model.fit( X_train, y_train )

# 4. 예측및 평가
from sklearn.metrics import accuracy_score

# 예측
y_pred = model.predict( X_test )
# 정답, 예측값을 넣어서 정확도 평가
accuracy_score( y_test, y_pred )

0.9649122807017544

- 결론
  - 프로토타입으로 베이스라인 구축을 통해 96.4%까지 진입

- 목표
  - AutoML이용하여 98%대까지 모델 성능 향상(교체)

# AutoML 적용 - 자동

### 데이터 준비

In [ ]:
# 1. 필요한 분류-모듈 모두 가져오기
from pycaret.classification import *

In [ ]:

# 2. 분할된 훈련용 데이터 (피처, 정답)를 하나로 합치기 => 파이캐럿에 넣기 위해서 구성
#   axis : 0이면 수직으로 병합(데이터가 늘어남), 1이면 수평으로 병합(컬럼 추가, 피처나 정답 데이터 추가)
# pandas의 concat() 함수는 데이터 단순 합치지기 함수
X = pd.concat( [X_train, y_train], axis=1 ) # 데이터의 수는 유지, 컬럼 확장
X.head(2)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
82,25.22,24.91,171.5,1878.0,0.1063,0.2665,0.3339,0.18450,0.1829,0.06782,...,33.62,211.7,2562.0,0.1573,0.6076,0.6476,0.2867,0.2355,0.1051,0
39,13.48,20.82,88.4,559.2,0.1016,0.1255,0.1063,0.05439,0.1720,0.06419,...,26.02,107.3,740.4,0.1610,0.4225,0.5030,0.2258,0.2807,0.1071,0


In [ ]:
# 데이터 512개, 31(피처 30개, 정답 1개)
X.shape

(512, 31)

### steup()
  - 피처 엔지니어링을 automl에 맡겨서 진행
    - 설정된 범위안에서만 작동
  - 직관적으로 피처 엔지니어링을 진행후 automl 진행 가능함.
    - automl은 알고리즘 및 파이프라인에만 집중되게 사용

In [ ]:
# 분류기 생성
# 파이캐럿의 setup() : automl 학습을 위한 설정(훈련데이터, 다양한 옵션설정)
clf = setup(
  data   = X,        # 피처 + 정답이 포함된 데이터, DataFrame
  target = 'target', # target 컬럼이 정답이다 => 컬럼명 지정
  verbose= True,     # 로그 출력
  # 데이터가 적어서 9:1로 설정
  train_size= 0.9,   # 훈련:검증 = 9:1 => 교차검증(cv) 수행하겠다 => 모델 과대/과소 체크,
  data_split_shuffle= True, # 데이터를 분할할때 섞겠다
  session_id= 100,   # 난수 시드
  normalize = True,  # 데이터에 대한 정규화 처리 진행(데이터를 조정하겠다) => 데이터의 스케일 조정
)

,Description,Value
0,Session id,100
1,Target,target
2,Target type,Binary
3,Original data shape,"(512, 31)"
4,Transformed data shape,"(512, 31)"
5,Transformed train set shape,"(460, 31)"
6,Transformed test set shape,"(52, 31)"
7,Numeric features,30
8,Preprocess,True
9,Imputation type,simple


In [ ]:
# 후보 알고리즘
models()

,Name,Reference,Turbo
ID,,,
lr,Logistic Regression,sklearn.linear_model._logistic.LogisticRegression,True
knn,K Neighbors Classifier,sklearn.neighbors._classification.KNeighborsCl...,True
nb,Naive Bayes,sklearn.naive_bayes.GaussianNB,True
dt,Decision Tree Classifier,sklearn.tree._classes.DecisionTreeClassifier,True
svm,SVM - Linear Kernel,sklearn.linear_model._stochastic_gradient.SGDC...,True
rbfsvm,SVM - Radial Kernel,sklearn.svm._classes.SVC,False
gpc,Gaussian Process Classifier,sklearn.gaussian_process._gpc.GaussianProcessC...,False
mlp,MLP Classifier,sklearn.neural_network._multilayer_perceptron....,False
ridge,Ridge Classifier,sklearn.linear_model._ridge.RidgeClassifier,True


### compare_models(), 후보 모델 획득

In [ ]:
# 최적 후보 상위 탑 5 획득
'''
- fold : 교차 검증은 5세트만 진행
- round : 교차 검증은 완료하는 라운드를 3번 진행
- sort : 상위 탑 5를 선택할대는 정확도를 기준으로 선정
- n_select : 상위 랭커 5개만 선정
'''
top_5 =  compare_models( fold=5, round=3, sort='Accuracy', n_select=5 )

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.972,0.995,0.979,0.976,0.977,0.940,0.941,1.642
et,Extra Trees Classifier,0.967,0.994,0.979,0.969,0.974,0.931,0.931,0.210
catboost,CatBoost Classifier,0.967,0.993,0.979,0.969,0.974,0.931,0.931,10.766
knn,K Neighbors Classifier,0.965,0.986,0.993,0.953,0.973,0.925,0.927,0.050
rf,Random Forest Classifier,0.963,0.989,0.979,0.962,0.970,0.921,0.922,0.264
qda,Quadratic Discriminant Analysis,0.959,0.994,0.965,0.969,0.967,0.913,0.914,0.040
gbc,Gradient Boosting Classifier,0.959,0.993,0.968,0.965,0.967,0.912,0.913,0.732
ridge,Ridge Classifier,0.957,0.996,0.996,0.939,0.966,0.905,0.910,0.040
ada,Ada Boost Classifier,0.957,0.988,0.968,0.962,0.965,0.908,0.909,0.206
xgboost,Extreme Gradient Boosting,0.957,0.992,0.965,0.966,0.965,0.907,0.909,0.134


Processing:   0%|          | 0/73 [00:00<?, ?it/s]

In [ ]:
# 상위 5개 모델 확인
# 상위 5개 모델 추출, 각 모델(알고리즘)별 (하이퍼) 파라미터 최적값 확인
top_5

[LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                    intercept_scaling=1, l1_ratio=None, max_iter=1000,
                    multi_class='auto', n_jobs=None, penalty='l2',
                    random_state=100, solver='lbfgs', tol=0.0001, verbose=0,
                    warm_start=False),
 ExtraTreesClassifier(bootstrap=False, ccp_alpha=0.0, class_weight=None,
                      criterion='gini', max_depth=None, max_features='sqrt',
                      max_leaf_nodes=None, max_samples=None,
                      min_impurity_decrease=0.0, min_samples_leaf=1,
                      min_samples_split=2, min_weight_fraction_leaf=0.0,
                      monotonic_cst=None, n_estimators=100, n_jobs=-1,
                      oob_score=False, random_state=100, verbose=0,
                      warm_start=False),
 KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
                      metric_params=None, n_jobs=-1, n_neighb

### blend_models() : 블랜딩 - 보팅
  - 여러 알고리즘을 섞어서 모델 구성
  - 앙상블 보팅 전략
    - 하드보팅(다수결), 소프트보팅(점유율, 총합으로결정)

In [ ]:
blended = blend_models( estimator_list = top_5,
              fold = 10, # cv 세트값, 하이퍼 파라미터값을 가지고 다시 진행한다는 의미
              optimize = 'Accuracy', # 최적화할 지표
              method = 'soft' # 소프트보팅
              )

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
1,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2,0.9783,1.0000,1.0000,0.9667,0.9831,0.9528,0.9538
3,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
4,0.9565,1.0000,1.0000,0.9355,0.9667,0.9044,0.9085
5,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
6,0.8913,0.9881,0.9643,0.8710,0.9153,0.7648,0.7726
7,0.9783,1.0000,0.9643,1.0000,0.9818,0.9548,0.9558
8,0.9565,0.9861,1.0000,0.9333,0.9655,0.9069,0.9108


Processing:   0%|          | 0/6 [00:00<?, ?it/s]

### finalize_model() : 최적 모델 추출

In [ ]:
model = finalize_model(blended)

In [ ]:
model

Pipeline(memory=Memory(location=None),
         steps=[('numerical_imputer',
                 TransformerWrapper(exclude=None,
                                    include=['mean radius', 'mean texture',
                                             'mean perimeter', 'mean area',
                                             'mean smoothness',
                                             'mean compactness',
                                             'mean concavity',
                                             'mean concave points',
                                             'mean symmetry',
                                             'mean fractal dimension',
                                             'radius error', 'texture error',
                                             'perimeter error', 'area error',
                                             'smoothness error',
                                             'com...
                                                                      max_features='sqrt',
                                                                      max_leaf_nodes=None,
                                                                      max_samples=None,
                                                                      min_impurity_decrease=0.0,
                                                                      min_samples_leaf=1,
                                                                      min_samples_split=2,
                                                                      min_weight_fraction_leaf=0.0,
                                                                      monotonic_cst=None,
                                                                      n_estimators=100,
                                                                      n_jobs=-1,
                                                                      oob_score=False,
                                                                      random_state=100,
                                                                      verbose=0,
                                                                      warm_start=False))],
                                  flatten_transform=True, n_jobs=-1,
                                  verbose=False, voting='soft',
                                  weights=None))],
         verbose=False)

### 예측 수행 및 평가

In [ ]:
y_pred = predict_model( model, data=X_test)

# 예측 결과 샘플 2개
y_pred.head(2)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,prediction_label,prediction_score
204,12.470000,18.600000,81.089996,481.899994,0.09965,0.1058,0.08005,0.03821,0.1925,0.06373,...,96.050003,677.900024,0.1426,0.2378,0.2671,0.1015,0.3014,0.08750,1,0.9130
70,18.940001,21.309999,123.599998,1130.000000,0.09009,0.1029,0.10800,0.07951,0.1582,0.05461,...,165.899994,1866.000000,0.1193,0.2336,0.2687,0.1789,0.2551,0.06589,0,0.9996


In [ ]:
# 평가 (실제정답, 예측값)
# 정확도가 동일한 부분은 체크해서 오후에 확인
accuracy_score(y_test, y_pred.prediction_label)

0.9649122807017544

# AutoML 적용 - 반자동(어느 정도 개입)

### 데이터준비(생략)

### setup() (생략)

### 모델 생성

In [ ]:
models()

,Name,Reference,Turbo
ID,,,
lr,Logistic Regression,sklearn.linear_model._logistic.LogisticRegression,True
knn,K Neighbors Classifier,sklearn.neighbors._classification.KNeighborsCl...,True
nb,Naive Bayes,sklearn.naive_bayes.GaussianNB,True
dt,Decision Tree Classifier,sklearn.tree._classes.DecisionTreeClassifier,True
svm,SVM - Linear Kernel,sklearn.linear_model._stochastic_gradient.SGDC...,True
rbfsvm,SVM - Radial Kernel,sklearn.svm._classes.SVC,False
gpc,Gaussian Process Classifier,sklearn.gaussian_process._gpc.GaussianProcessC...,False
mlp,MLP Classifier,sklearn.neural_network._multilayer_perceptron....,False
ridge,Ridge Classifier,sklearn.linear_model._ridge.RidgeClassifier,True


In [ ]:
# 모델 6개 구성 -> 경쟁을 통해서 top 모덿 획득이 아닌 직관으로 모델 지정
# 모든 모델을 검토하지 않아도 됨 => 시간 절약, 반면에 좋은 모델은 놓칠수 있다
model_gbc = create_model('gbc', fold=5)
model_xgboost = create_model('xgboost', fold=5)
model_catboost = create_model('catboost', fold=5)
model_lightgbm = create_model('lightgbm', fold=5)
model_lr = create_model('lr', fold=5)
model_lda = create_model('lda', fold=5)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9891,1.0000,0.9825,1.0000,0.9912,0.9771,0.9773
1,0.9783,0.9985,0.9825,0.9825,0.9825,0.9539,0.9539
2,0.9674,0.9975,0.9825,0.9655,0.9739,0.9304,0.9307
3,0.9239,0.9900,0.9298,0.9464,0.9381,0.8395,0.8397
4,0.9348,0.9784,0.9649,0.9322,0.9483,0.8601,0.8610
Mean,0.9587,0.9929,0.9684,0.9653,0.9668,0.9122,0.9125
Std,0.0252,0.0080,0.0205,0.0243,0.0203,0.0534,0.0533


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9674,0.9980,0.9474,1.0000,0.9730,0.9320,0.9341
1,0.9674,0.9975,0.9649,0.9821,0.9735,0.9312,0.9315
2,0.9565,0.9980,0.9825,0.9492,0.9655,0.9067,0.9077
3,0.9674,0.9935,0.9649,0.9821,0.9735,0.9312,0.9315
4,0.9239,0.9714,0.9649,0.9167,0.9402,0.8359,0.8380
Mean,0.9565,0.9917,0.9649,0.9660,0.9651,0.9074,0.9085
Std,0.0168,0.0103,0.0111,0.0296,0.0128,0.0370,0.0366


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9783,1.0000,0.9649,1.0000,0.9821,0.9544,0.9554
1,0.9891,0.9990,1.0000,0.9828,0.9913,0.9768,0.9771
2,0.9674,0.9980,1.0000,0.9500,0.9744,0.9297,0.9320
3,0.9565,0.9935,0.9649,0.9649,0.9649,0.9078,0.9078
4,0.9457,0.9724,0.9649,0.9483,0.9565,0.8841,0.8843
Mean,0.9674,0.9926,0.9789,0.9692,0.9738,0.9305,0.9313
Std,0.0154,0.0103,0.0172,0.0198,0.0123,0.0328,0.0330


...

Processing:   0%|          | 0/4 [00:00<?, ?it/s]

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9783,1.0000,0.9649,1.0000,0.9821,0.9544,0.9554
1,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2,0.9783,0.9945,0.9825,0.9825,0.9825,0.9539,0.9539
3,0.9457,0.9970,0.9474,0.9643,0.9558,0.8853,0.8856
4,0.9565,0.9845,1.0000,0.9344,0.9661,0.9057,0.9097
Mean,0.9717,0.9952,0.9789,0.9762,0.9773,0.9399,0.9409
Std,0.0190,0.0057,0.0205,0.0247,0.0152,0.0404,0.0398


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9674,0.9990,0.9825,0.9655,0.9739,0.9304,0.9307
1,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2,0.9348,0.9990,1.0000,0.9048,0.9500,0.8569,0.8658
3,0.9348,0.9940,1.0000,0.9048,0.9500,0.8569,0.8658
4,0.9348,0.9870,1.0000,0.9048,0.9500,0.8569,0.8658
Mean,0.9543,0.9958,0.9965,0.9360,0.9648,0.9002,0.9056
Std,0.0261,0.0049,0.0070,0.0397,0.0199,0.0574,0.0535


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

### 튜닝

- 개별 모델의 성능 극대화

In [ ]:
# choose_better : 성능이 높아지는쪽으로 선택
tune_model_gbc      = tune_model(  model_gbc, fold=5, optimize='Accuracy', choose_better=True )

tune_model_xgboost  = tune_model(  model_xgboost, fold=5, optimize='Accuracy', choose_better=True )

tune_model_lightgbm = tune_model(  model_lightgbm, fold=5, optimize='Accuracy', choose_better=True )

tune_model_catboost = tune_model(  model_catboost, fold=5, optimize='Accuracy', choose_better=True )

tune_model_lr       = tune_model(  model_lr, fold=5, optimize='Accuracy', choose_better=True )

tune_model_lda      = tune_model(  model_lda, fold=5, optimize='Accuracy', choose_better=True )

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9891,0.9995,0.9825,1.0000,0.9912,0.9771,0.9773
1,0.9674,0.9970,0.9649,0.9821,0.9735,0.9312,0.9315
2,0.9783,0.9930,1.0000,0.9661,0.9828,0.9534,0.9544
3,0.9457,0.9910,0.9474,0.9643,0.9558,0.8853,0.8856
4,0.9348,0.9677,0.9649,0.9322,0.9483,0.8601,0.8610
Mean,0.9630,0.9896,0.9719,0.9689,0.9703,0.9214,0.9220
Std,0.0202,0.0114,0.0179,0.0224,0.0161,0.0431,0.0430


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9891,1.0000,1.0000,0.9828,0.9913,0.9768,0.9771
1,0.9783,0.9990,1.0000,0.9661,0.9828,0.9534,0.9544
2,0.9674,0.9995,1.0000,0.9500,0.9744,0.9297,0.9320
3,0.9457,0.9935,0.9825,0.9333,0.9573,0.8828,0.8850
4,0.9457,0.9835,1.0000,0.9194,0.9580,0.8814,0.8877
Mean,0.9652,0.9951,0.9965,0.9503,0.9727,0.9248,0.9272
Std,0.0174,0.0063,0.0070,0.0226,0.0135,0.0379,0.0363


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

...

Fitting 5 folds for each of 10 candidates, totalling 50 fits


Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9783,0.9985,0.9825,0.9825,0.9825,0.9539,0.9539
1,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2,0.9348,0.9980,1.0000,0.9048,0.9500,0.8569,0.8658
3,0.9457,0.9940,1.0000,0.9194,0.9580,0.8814,0.8877
4,0.9457,0.9769,1.0000,0.9194,0.9580,0.8814,0.8877
Mean,0.9609,0.9935,0.9965,0.9452,0.9697,0.9147,0.9190
Std,0.0244,0.0085,0.0070,0.0384,0.0187,0.0536,0.0501


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 5 folds for each of 10 candidates, totalling 50 fits


### 최종 후보 모델

In [ ]:
# 우의 모델 전부 사용하거나 그중 상위권만 사용등 다양한 구성 가능함
candidate_models = [
    tune_model_gbc,
    tune_model_xgboost,
    tune_model_lightgbm,
    tune_model_catboost,
    tune_model_lr,
    tune_model_lda,
]
# 의미적으로는 top_5와 a동일한 성격이므로, 이후 절차는 동일함

### 이후 절차 동일( 블랜딩, 최적모델, 예측수행 및 평가)